# 05 分層分析與交絡因子

Ch03 發現淋浴使用者的 RR 偏高，但資深疫調人員懷疑是**交絡（confounding）**造成的。

這堂課：**驗證交絡條件 → 分層 RR → 森林圖 → Mantel-Haenszel 調整 → 同質性檢定**。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

# -- CJK font setup (避免中文標籤顯示為方框) --
plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")

In [ ]:
# --- Step 2: 粗 RR 回顧 ---
ct = pd.crosstab(df["shower_use"], df["infected"])
a = int(ct.loc[1, 1])
b = int(ct.loc[1, 0])
c = int(ct.loc[0, 1])
d = int(ct.loc[0, 0])

crude_rr = risk_ratio(a, a + b, c, c + d)
print(f"粗 RR (shower_use → infected) = {crude_rr:.3f}")
print(f"  暴露組風險: {a/(a+b):.1%}  未暴露組風險: {c/(c+d):.1%}")

## DAG（有向無環圖）

懷疑 `functional_status`（功能狀態）是交絡因子：

```
functional_status → shower_use → infected
functional_status ─────────────→ infected
```

- 能自主行走的人才用淋浴（`functional_status → shower_use`）
- 功能狀態好的人活動範圍大、暴露機會多（`functional_status → infected`）

In [ ]:
# --- Step 3: 驗證交絡三要件 ---

# 要件 1：functional_status 與 shower_use 有關聯？
print("=== 功能狀態 × 淋浴使用率 ===")
print(pd.crosstab(df["functional_status"], df["shower_use"],
                  margins=True, normalize="index").round(3))

print()

# 要件 2：functional_status 與 infected 有關聯？
print("=== 功能狀態 × 感染率 ===")
print(pd.crosstab(df["functional_status"], df["infected"],
                  margins=True, normalize="index").round(3))

In [ ]:
# --- Step 4: 分層分析 ---
# 按 functional_status 分層，各層分別計算 shower_use 的 RR

strata = sorted(df["functional_status"].unique())
stratum_results = []

for s in strata:
    sub = df[df["functional_status"] == s]
    ct_s = pd.crosstab(sub["shower_use"], sub["infected"])

    if ct_s.shape != (2, 2):
        print(f"  {s}: 跳過（缺少某些組合）")
        continue

    a_s = int(ct_s.loc[1, 1])
    b_s = int(ct_s.loc[1, 0])
    c_s = int(ct_s.loc[0, 1])
    d_s = int(ct_s.loc[0, 0])
    n_s = a_s + b_s + c_s + d_s

    rr_s = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)

    # 95% CI
    ln_rr = np.log(rr_s)
    se = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
    ci_lo = np.exp(ln_rr - 1.96 * se)
    ci_hi = np.exp(ln_rr + 1.96 * se)

    stratum_results.append({
        "stratum": s, "n": n_s,
        "a": a_s, "b": b_s, "c": c_s, "d": d_s,
        "RR": rr_s, "CI_lower": ci_lo, "CI_upper": ci_hi,
    })

results_df = pd.DataFrame(stratum_results)
print("=== 分層 RR ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR={row['RR']:.3f}  "
          f"(95% CI: {row['CI_lower']:.3f}–{row['CI_upper']:.3f})  n={row['n']}")
print(f"\n  粗 RR = {crude_rr:.3f}")

In [ ]:
# --- Step 5: 森林圖 ---
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(results_df))

ax.errorbar(
    results_df["RR"], y_pos,
    xerr=[results_df["RR"] - results_df["CI_lower"],
          results_df["CI_upper"] - results_df["RR"]],
    fmt="o", color="#2c7fb8", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=crude_rr, color="red", linestyle=":", alpha=0.7,
           label=f"粗 RR={crude_rr:.2f}")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(results_df["stratum"])
ax.set_xlabel("Risk Ratio (RR)")
ax.set_title("分層分析森林圖：淋浴使用 → 感染（按功能狀態分層）")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 6: Mantel-Haenszel 加權 RR ---
numerator = 0
denominator = 0

for _, row in results_df.iterrows():
    a_i, b_i, c_i, d_i = row["a"], row["b"], row["c"], row["d"]
    n_i = a_i + b_i + c_i + d_i
    numerator += a_i * (c_i + d_i) / n_i
    denominator += c_i * (a_i + b_i) / n_i

rr_mh = numerator / denominator

print(f"Mantel-Haenszel 調整後 RR = {rr_mh:.3f}")
print(f"粗 RR                     = {crude_rr:.3f}")
print(f"差異                      = {crude_rr - rr_mh:.3f}")

if crude_rr - rr_mh > 0.1:
    print("\n→ 粗 RR 被交絡膨脹了！控制功能狀態後，淋浴的效應變小。")
else:
    print("\n→ 控制功能狀態後，RR 變化不大，交絡效應有限。")

In [ ]:
# --- Step 7: 同質性檢定 ---
# 各層 RR 差異大 → 可能有交互作用（effect modification）

rr_values = results_df["RR"].values
rr_range = rr_values.max() - rr_values.min()

print("=== 同質性評估 ===")
for _, row in results_df.iterrows():
    print(f"  {row['stratum']:20s}  RR = {row['RR']:.3f}")
print(f"\n  RR 範圍：{rr_values.min():.3f} – {rr_values.max():.3f}")
print(f"  變異幅度：{rr_range:.3f}")

if rr_range > 0.5:
    print("\n→ 各層 RR 差異較大，可能存在效果修飾（effect modification）")
    print("  建議分層報告，不宜只報告合併的 RR_MH")
else:
    print("\n→ 各層 RR 相近，可合理使用 MH 加權合併值")

In [ ]:
# --- Step 8: 第二個範例 — 按樓層分層 ---
print("=== 按樓層分層分析：淋浴 → 感染 ===")

for floor in sorted(df["floor"].unique()):
    sub = df[df["floor"] == floor]
    ct_f = pd.crosstab(sub["shower_use"], sub["infected"])
    if ct_f.shape != (2, 2):
        print(f"  {floor}F: 跳過")
        continue
    a_f, b_f = int(ct_f.loc[1, 1]), int(ct_f.loc[1, 0])
    c_f, d_f = int(ct_f.loc[0, 1]), int(ct_f.loc[0, 0])
    rr_f = risk_ratio(a_f, a_f + b_f, c_f, c_f + d_f)
    print(f"  {floor}F: RR={rr_f:.3f}  "
          f"(shower: {a_f}/{a_f+b_f}, no shower: {c_f}/{c_f+d_f})")

print(f"\n  粗 RR = {crude_rr:.3f}")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 交絡三要件 | 驗證 C-暴露、C-結果的關聯 |
| 分層 RR | `crosstab` + 迴圈計算各層 RR |
| 森林圖 | `errorbar` 視覺化各層效應 |
| MH 調整 | 手算 Mantel-Haenszel RR |
| 同質性 | 各層 RR 比較 → 交互作用判斷 |

**限制**：分層分析一次只能控制一個交絡因子。如果同時有年齡、功能狀態、共病等多個交絡呢？
→ Ch06 的**邏輯斯迴歸**可以一次調整所有變項。